# SurfaceTarget-CRC — phase 01 — patch 1 4

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.4
#
# RNA-seq Metadata Quality Audit
# and MSI/MSS Clinical Annotation Discovery
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import requests
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

INVENTORY_PATH = (
    METADATA_DIR / "tcga_rnaseq_file_inventory.csv"
)

CBIO_API = "https://www.cbioportal.org/api"

STUDY_ID = "coadread_tcga_pan_can_atlas_2018"

# ------------------------------------------------------------
# 2. Load existing RNA-seq inventory
# ------------------------------------------------------------

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.4 — MSI ANNOTATION DISCOVERY")
print("=" * 65)

if not INVENTORY_PATH.exists():
    raise FileNotFoundError(
        "RNA-seq inventory not found. "
        "Please run Patch 1.3 first."
    )

inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nRNA-seq inventory loaded successfully.")

print(f"Metadata rows: {len(inventory)}")

print(
    f"Unique files: "
    f"{inventory['file_id'].nunique()}"
)

print(
    f"Unique patients: "
    f"{inventory['patient_id'].nunique()}"
)

# ------------------------------------------------------------
# 3. Audit file-to-sample relationships
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RNA-SEQ METADATA QUALITY AUDIT")
print("-" * 65)

file_sample_counts = (
    inventory.groupby("file_id")["sample_id"]
    .nunique()
)

sample_file_counts = (
    inventory[
        inventory["sample_id"] != ""
    ]
    .groupby("sample_id")["file_id"]
    .nunique()
)

multi_sample_files = int(
    (file_sample_counts > 1).sum()
)

multi_file_samples = int(
    (sample_file_counts > 1).sum()
)

print(
    "\nFiles associated with multiple sample IDs:",
    multi_sample_files
)

print(
    "Samples associated with multiple files:",
    multi_file_samples
)

print(
    "Maximum files associated with one sample:",
    sample_file_counts.max()
)

print("\nSamples with multiple files (first 10):")

print(
    sample_file_counts[
        sample_file_counts > 1
    ]
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# 4. Connect to cBioPortal
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CBIOPORTAL CLINICAL METADATA")
print("-" * 65)

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

attributes_url = (
    f"{CBIO_API}/studies/"
    f"{STUDY_ID}/clinical-attributes"
)

response = session.get(
    attributes_url,
    params={"pageSize": 10000},
    timeout=60
)

response.raise_for_status()

attributes = response.json()

if not isinstance(attributes, list):
    raise RuntimeError(
        "Unexpected clinical attribute API response."
    )

print("\nStudy:", STUDY_ID)

print(
    "Total available clinical attributes:",
    len(attributes)
)

# ------------------------------------------------------------
# 5. Find MSI-related clinical attributes
# ------------------------------------------------------------

keywords = [
    "MSI",
    "MANTIS",
    "MICROSAT",
    "MSISENSOR",
    "MSI SENSOR"
]

msi_attributes = []

for attribute in attributes:

    searchable_text = " ".join([
        str(attribute.get("id", "")),
        str(attribute.get("displayName", "")),
        str(attribute.get("name", ""))
    ]).upper()

    if any(
        keyword in searchable_text
        for keyword in keywords
    ):
        msi_attributes.append(attribute)

print("\nMSI-related clinical attributes found:")

if not msi_attributes:
    print("No MSI-related attributes identified.")

for attribute in msi_attributes:

    print(
        f"\nID: {attribute.get('id')}"
    )

    print(
        "Name:",
        attribute.get(
            "displayName",
            attribute.get("name", "Unknown")
        )
    )

    print(
        "Patient-level attribute:",
        attribute.get("patientAttribute")
    )

# ------------------------------------------------------------
# 6. Retrieve MSI-related clinical measurements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RETRIEVING MSI CLINICAL MEASUREMENTS")
print("-" * 65)

all_clinical_records = []

for attribute in msi_attributes:

    attribute_id = attribute.get("id")

    clinical_level = (
        "PATIENT"
        if attribute.get("patientAttribute", False)
        else "SAMPLE"
    )

    endpoint = (
        f"{CBIO_API}/studies/"
        f"{STUDY_ID}/clinical-data"
    )

    params = {
        "clinicalDataType": clinical_level,
        "attributeId": attribute_id,
        "pageSize": 10000
    }

    print(f"\nRetrieving: {attribute_id}")

    try:

        response = session.get(
            endpoint,
            params=params,
            timeout=90
        )

        response.raise_for_status()

        records = response.json()

        if not isinstance(records, list):
            raise ValueError(
                "Unexpected clinical data format."
            )

        all_clinical_records.extend(records)

        print(
            f"Records retrieved: {len(records)}"
        )

    except Exception as e:

        print(
            f"Retrieval failed for {attribute_id}: {e}"
        )

# ------------------------------------------------------------
# 7. Save clinical annotation data
# ------------------------------------------------------------

clinical_df = pd.DataFrame(
    all_clinical_records
)

clinical_path = (
    METADATA_DIR /
    "tcga_pancancer_msi_clinical_raw.csv"
)

clinical_df.to_csv(
    clinical_path,
    index=False
)

# ------------------------------------------------------------
# 8. Inspect MSI measurements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSI CLINICAL DATA SUMMARY")
print("-" * 65)

if clinical_df.empty:

    print(
        "\nNo MSI clinical measurements retrieved."
    )

else:

    print(
        "\nTotal clinical records:",
        len(clinical_df)
    )

    print(
        "\nAvailable columns:",
        clinical_df.columns.tolist()
    )

    if "clinicalAttributeId" in clinical_df.columns:

        for attribute_id, group in clinical_df.groupby(
            "clinicalAttributeId"
        ):

            print(f"\nAttribute: {attribute_id}")

            print(f"Records: {len(group)}")

            if "value" in group.columns:

                values = group["value"].dropna()

                numeric_values = pd.to_numeric(
                    values,
                    errors="coerce"
                )

                numeric_fraction = (
                    numeric_values.notna().mean()
                    if len(values) else 0
                )

                if numeric_fraction > 0.85:

                    print(
                        "Numeric observations:",
                        numeric_values.notna().sum()
                    )

                    print(
                        "Minimum:",
                        numeric_values.min()
                    )

                    print(
                        "Median:",
                        numeric_values.median()
                    )

                    print(
                        "Maximum:",
                        numeric_values.max()
                    )

                else:

                    print(
                        "\nMost common values:"
                    )

                    print(
                        values.value_counts()
                        .head(10)
                        .to_string()
                    )

# ------------------------------------------------------------
# 9. Check patient identifier overlap
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GDC / CBIOPORTAL PATIENT OVERLAP")
print("-" * 65)

gdc_patients = set(
    inventory["patient_id"].dropna()
)

if (
    not clinical_df.empty
    and "patientId" in clinical_df.columns
):

    cbio_patients = set(
        clinical_df["patientId"].dropna()
    )

    overlap = (
        gdc_patients.intersection(
            cbio_patients
        )
    )

    print(
        "\nGDC RNA-seq patients:",
        len(gdc_patients)
    )

    print(
        "Patients with retrieved MSI annotations:",
        len(cbio_patients)
    )

    print(
        "Overlapping patients:",
        len(overlap)
    )

else:

    overlap = set()

    print(
        "\nPatient overlap could not be evaluated."
    )

# ------------------------------------------------------------
# 10. Save provenance metadata
# ------------------------------------------------------------

metadata = {
    "project": "SurfaceTarget-CRC",

    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "cBioPortal",

    "study_id": STUDY_ID,

    "api": CBIO_API,

    "msi_attributes": [
        a.get("id")
        for a in msi_attributes
    ],

    "gdc_patients": len(gdc_patients),

    "overlapping_patients": len(overlap),

    "files_with_multiple_samples": multi_sample_files,

    "samples_with_multiple_files": multi_file_samples,

    "msi_status_assigned": False,

    "note": (
        "Exploratory clinical annotation retrieval. "
        "No MSI/MSS classification thresholds "
        "have been applied."
    )
}

metadata_path = (
    METADATA_DIR /
    "tcga_msi_annotation_discovery.json"
)

with open(metadata_path, "w") as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print("\n" + "=" * 65)
print("FILES SAVED")
print("=" * 65)

print("\n1. tcga_pancancer_msi_clinical_raw.csv")
print("2. tcga_msi_annotation_discovery.json")

print("\nNo MSS labels assigned yet.")
print("No RNA-seq expression data downloaded.")

print("\nPATCH 1.4 COMPLETE")